# 04 · Test runs

All methods on the **test** split (118 queries) with the settings chosen in `03_dev_tuning` (`v2/config.json`).
- **Candidate order** is shuffled per query and seed.
- **Seeds.** Order-sensitive LLM methods run with 5 shuffle seeds: 3 core seeds first, so a complete result exists early, then 2 extra seeds at the end. The other methods are order-independent; their seed only breaks ties.
- **Speed.** The LLM work is prompt-bound (1–5k tokens per call), so concurrent requests did not speed it up on this machine. They gave identical outputs in a 12-query check, but runs stay sequential.
- **Saving.** Runs are written one query at a time, so an interrupted run resumes where it stopped.

Run in the background with papermill, which saves this notebook after every cell:

```bash
cd research && .venv/bin/papermill v2/04_test_runs.ipynb v2/04_test_runs.ipynb --log-output
```

In [1]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", message="IProgress not found")

RESEARCH = Path.cwd().parent if Path.cwd().name == "v2" else Path.cwd()
sys.path.insert(0, str(RESEARCH))

import pandas as pd
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

from src.figures import save, setup_style
setup_style()

In [2]:
BENCHES = ["concept", "random"]
CORE_SEEDS = [0, 1, 2]      # complete result first: every method, both benchmarks
EXTRA_SEEDS = [3, 4]        # then more shuffle seeds for the order-sensitive LLM methods
SPLIT = "test"
POSITION_BIAS_SEEDS = [0]

In [3]:
import json
from src.pipeline_v2 import RunConfig, run_methods

C = json.loads((RESEARCH / "v2" / "config.json").read_text())
def cfg(bench, **kw):
    base = dict(bench=bench, fields=C["fields"], max_chars=C["max_chars"], query_max_chars=C["query_max_chars"],
                split=SPLIT, stage1_k=C["stage1_k"])
    return RunConfig(**(base | kw))
C

{'fields': 'tac',
 'max_chars': 600,
 'query_max_chars': 1500,
 'stage1_k': 10,
 'chosen_on': 'dev',
 'dev_mean_rank': {'tac_600_1500': 1.75,
  'ta_600_1500': 1.875,
  't_600_1500': 2.375}}

### Lexical, embedding and cross-encoder methods

In [4]:
for bench in BENCHES:
    display(run_methods(["bm25", "tfidf", "minilm", "specter2", "qwen", "embeddinggemma", "qwen+minilm", "bge-reranker"],
                        cfg(bench, seeds=[0])))

,bench,text_tag,method,order,seed,queries,mrr,rprec,ndcg,ap,top1,ok,partial,failed,sec_per_query
0,concept,tac_600_1500,bge-reranker,shuffled,0,118,0.872,58.118,0.631,0.687,0.797,118,0,0,0.386
1,concept,tac_600_1500,bm25,shuffled,0,118,0.856,63.351,0.670,0.713,0.746,118,0,0,0.002
2,concept,tac_600_1500,embeddinggemma,shuffled,0,118,0.983,78.457,0.832,0.872,0.966,118,0,0,0.000
3,concept,tac_600_1500,minilm,shuffled,0,118,0.956,75.727,0.802,0.848,0.924,118,0,0,0.000
4,concept,tac_600_1500,qwen,shuffled,0,118,0.970,80.075,0.839,0.876,0.941,118,0,0,0.000
5,concept,tac_600_1500,qwen+minilm,shuffled,0,118,0.957,78.152,0.823,0.868,0.924,118,0,0,0.001
6,concept,tac_600_1500,specter2,shuffled,0,118,0.947,76.637,0.805,0.848,0.907,118,0,0,0.000
7,concept,tac_600_1500,tfidf,shuffled,0,118,0.880,65.423,0.694,0.745,0.788,118,0,0,0.002


,bench,text_tag,method,order,seed,queries,mrr,rprec,ndcg,ap,top1,ok,partial,failed,sec_per_query
0,random,tac_600_1500,bge-reranker,shuffled,0,118,0.912,67.241,0.717,0.761,0.847,118,0,0,0.408
1,random,tac_600_1500,bm25,shuffled,0,118,0.904,68.551,0.736,0.785,0.822,118,0,0,0.002
2,random,tac_600_1500,embeddinggemma,shuffled,0,118,1.000,85.618,0.892,0.922,1.000,118,0,0,0.000
3,random,tac_600_1500,minilm,shuffled,0,118,0.985,83.308,0.870,0.910,0.975,118,0,0,0.000
4,random,tac_600_1500,qwen,shuffled,0,118,0.987,86.976,0.900,0.929,0.975,118,0,0,0.000
5,random,tac_600_1500,qwen+minilm,shuffled,0,118,0.986,85.626,0.888,0.922,0.975,118,0,0,0.000
6,random,tac_600_1500,specter2,shuffled,0,118,0.976,83.478,0.868,0.906,0.958,118,0,0,0.000
7,random,tac_600_1500,tfidf,shuffled,0,118,0.921,75.317,0.787,0.828,0.864,118,0,0,0.003


### Listwise LLM: full list and sliding window (3 core seeds)

In [5]:
for bench in BENCHES:
    display(run_methods(["llama", "llama-window"], cfg(bench, seeds=CORE_SEEDS)))

,bench,text_tag,method,order,seed,queries,mrr,rprec,ndcg,ap,top1,ok,partial,failed,sec_per_query
0,concept,tac_600_1500,llama,shuffled,0,118,0.608,43.104,0.433,0.501,0.356,74,44,0,3.824
1,concept,tac_600_1500,llama,shuffled,1,118,0.675,43.271,0.455,0.529,0.483,75,43,0,3.640
2,concept,tac_600_1500,llama,shuffled,2,118,0.648,42.374,0.442,0.513,0.449,71,47,0,3.905
3,concept,tac_600_1500,llama-window,shuffled,0,118,0.787,54.317,0.577,0.618,0.627,117,1,0,6.116
4,concept,tac_600_1500,llama-window,shuffled,1,118,0.813,54.360,0.589,0.637,0.678,117,1,0,6.047
5,concept,tac_600_1500,llama-window,shuffled,2,118,0.776,50.226,0.547,0.612,0.636,116,2,0,6.073


,bench,text_tag,method,order,seed,queries,mrr,rprec,ndcg,ap,top1,ok,partial,failed,sec_per_query
0,random,tac_600_1500,llama,shuffled,0,118,0.663,46.326,0.478,0.541,0.441,75,41,2,3.706
1,random,tac_600_1500,llama,shuffled,1,118,0.721,43.361,0.467,0.531,0.542,76,41,1,3.580
2,random,tac_600_1500,llama,shuffled,2,118,0.672,45.548,0.472,0.536,0.475,70,48,0,3.585
3,random,tac_600_1500,llama-window,shuffled,0,118,0.794,56.141,0.601,0.647,0.636,116,2,0,5.793
4,random,tac_600_1500,llama-window,shuffled,1,118,0.808,55.689,0.600,0.652,0.669,118,0,0,5.825
5,random,tac_600_1500,llama-window,shuffled,2,118,0.815,57.623,0.621,0.669,0.669,116,2,0,5.848


### Llama + MiniLM fusion (derived from the listwise runs, no extra LLM calls)

In [6]:
for bench in BENCHES:
    display(run_methods(["llama+minilm"], cfg(bench, seeds=CORE_SEEDS)))

,bench,text_tag,method,order,seed,queries,mrr,rprec,ndcg,ap,top1,ok,partial,failed,sec_per_query
0,concept,tac_600_1500,llama+minilm,shuffled,0,118,0.935,64.009,0.700,0.750,0.890,74,44,0,0.003
1,concept,tac_600_1500,llama+minilm,shuffled,1,118,0.947,66.790,0.724,0.768,0.915,75,43,0,0.002
2,concept,tac_600_1500,llama+minilm,shuffled,2,118,0.924,63.764,0.697,0.749,0.873,71,47,0,0.002


,bench,text_tag,method,order,seed,queries,mrr,rprec,ndcg,ap,top1,ok,partial,failed,sec_per_query
0,random,tac_600_1500,llama+minilm,shuffled,0,118,0.966,70.465,0.764,0.808,0.941,75,41,2,0.003
1,random,tac_600_1500,llama+minilm,shuffled,1,118,0.973,68.177,0.751,0.806,0.949,76,41,1,0.002
2,random,tac_600_1500,llama+minilm,shuffled,2,118,0.977,69.072,0.756,0.809,0.958,70,48,0,0.002


### Two-stage Qwen3 → Llama (3 core seeds; stage-2 input shuffled)

In [7]:
for bench in BENCHES:
    display(run_methods(["qwen→llama"], cfg(bench, seeds=CORE_SEEDS)))

,bench,text_tag,method,order,seed,queries,mrr,rprec,ndcg,ap,top1,ok,partial,failed,sec_per_query
0,concept,tac_600_1500,qwen→llama_k10_shuffled,shuffled,0,118,0.869,69.401,0.720,0.774,0.763,118,0,0,1.718
1,concept,tac_600_1500,qwen→llama_k10_shuffled,shuffled,1,118,0.839,69.117,0.709,0.769,0.703,118,0,0,1.718
2,concept,tac_600_1500,qwen→llama_k10_shuffled,shuffled,2,118,0.842,71.009,0.722,0.771,0.712,118,0,0,1.716


,bench,text_tag,method,order,seed,queries,mrr,rprec,ndcg,ap,top1,ok,partial,failed,sec_per_query
0,random,tac_600_1500,qwen→llama_k10_shuffled,shuffled,0,118,0.907,74.148,0.771,0.828,0.831,118,0,0,1.723
1,random,tac_600_1500,qwen→llama_k10_shuffled,shuffled,1,118,0.881,72.954,0.757,0.816,0.780,118,0,0,1.723
2,random,tac_600_1500,qwen→llama_k10_shuffled,shuffled,2,118,0.861,75.078,0.764,0.814,0.737,117,1,0,1.723


### Pointwise LLM (order-independent, 1 seed)

In [8]:
for bench in BENCHES:
    display(run_methods(["llama-pointwise"], cfg(bench, seeds=[0])))

,bench,text_tag,method,order,seed,queries,mrr,rprec,ndcg,ap,top1,ok,partial,failed,sec_per_query
0,concept,tac_600_1500,llama-pointwise,shuffled,0,118,0.928,73.934,0.775,0.813,0.873,118,0,0,4.245


,bench,text_tag,method,order,seed,queries,mrr,rprec,ndcg,ap,top1,ok,partial,failed,sec_per_query
0,random,tac_600_1500,llama-pointwise,shuffled,0,118,0.947,78.756,0.817,0.852,0.898,118,0,0,4.241


### Position-bias conditions

Listwise Llama on the same test queries, with candidates in v1's positives-first order and in reversed order. Analysed in `06_analysis`.

In [9]:
for bench in BENCHES:
    display(run_methods(["llama"], cfg(bench, orders=["original", "reversed"], seeds=POSITION_BIAS_SEEDS)))

,bench,text_tag,method,order,seed,queries,mrr,rprec,ndcg,ap,top1,ok,partial,failed,sec_per_query
0,concept,tac_600_1500,llama,original,0,118,0.979,74.066,0.807,0.819,0.975,61,56,1,3.588
1,concept,tac_600_1500,llama,reversed,0,118,0.274,19.899,0.197,0.371,0.169,77,40,1,3.603
2,concept,tac_600_1500,llama,shuffled,0,118,0.608,43.104,0.433,0.501,0.356,74,44,0,3.824
3,concept,tac_600_1500,llama,shuffled,1,118,0.675,43.271,0.455,0.529,0.483,75,43,0,3.640
4,concept,tac_600_1500,llama,shuffled,2,118,0.648,42.374,0.442,0.513,0.449,71,47,0,3.905


,bench,text_tag,method,order,seed,queries,mrr,rprec,ndcg,ap,top1,ok,partial,failed,sec_per_query
0,random,tac_600_1500,llama,original,0,118,0.947,72.295,0.783,0.805,0.941,66,52,0,3.641
1,random,tac_600_1500,llama,reversed,0,118,0.334,29.988,0.287,0.438,0.229,78,39,1,3.573
2,random,tac_600_1500,llama,shuffled,0,118,0.663,46.326,0.478,0.541,0.441,75,41,2,3.706
3,random,tac_600_1500,llama,shuffled,1,118,0.721,43.361,0.467,0.531,0.542,76,41,1,3.580
4,random,tac_600_1500,llama,shuffled,2,118,0.672,45.548,0.472,0.536,0.475,70,48,0,3.585


### Extra seeds (3 and 4) for the order-sensitive LLM methods

In [10]:
for bench in BENCHES:
    display(run_methods(["llama", "llama-window", "llama+minilm", "qwen→llama"], cfg(bench, seeds=EXTRA_SEEDS)))

,bench,text_tag,method,order,seed,queries,mrr,rprec,ndcg,ap,top1,ok,partial,failed,sec_per_query
0,concept,tac_600_1500,llama,original,0,118,0.979,74.066,0.807,0.819,0.975,61,56,1,3.588
1,concept,tac_600_1500,llama,reversed,0,118,0.274,19.899,0.197,0.371,0.169,77,40,1,3.603
2,concept,tac_600_1500,llama,shuffled,0,118,0.608,43.104,0.433,0.501,0.356,74,44,0,3.824
3,concept,tac_600_1500,llama,shuffled,1,118,0.675,43.271,0.455,0.529,0.483,75,43,0,3.640
4,concept,tac_600_1500,llama,shuffled,2,118,0.648,42.374,0.442,0.513,0.449,71,47,0,3.905
5,concept,tac_600_1500,llama,shuffled,3,118,0.618,41.102,0.418,0.485,0.356,66,52,0,3.492
6,concept,tac_600_1500,llama,shuffled,4,118,0.640,40.296,0.417,0.493,0.424,74,44,0,3.517
7,concept,tac_600_1500,llama+minilm,shuffled,0,118,0.935,64.009,0.700,0.750,0.890,74,44,0,0.003
8,concept,tac_600_1500,llama+minilm,shuffled,1,118,0.947,66.790,0.724,0.768,0.915,75,43,0,0.002
9,concept,tac_600_1500,llama+minilm,shuffled,2,118,0.924,63.764,0.697,0.749,0.873,71,47,0,0.002


,bench,text_tag,method,order,seed,queries,mrr,rprec,ndcg,ap,top1,ok,partial,failed,sec_per_query
0,random,tac_600_1500,llama,original,0,118,0.947,72.295,0.783,0.805,0.941,66,52,0,3.641
1,random,tac_600_1500,llama,reversed,0,118,0.334,29.988,0.287,0.438,0.229,78,39,1,3.573
2,random,tac_600_1500,llama,shuffled,0,118,0.663,46.326,0.478,0.541,0.441,75,41,2,3.706
3,random,tac_600_1500,llama,shuffled,1,118,0.721,43.361,0.467,0.531,0.542,76,41,1,3.580
4,random,tac_600_1500,llama,shuffled,2,118,0.672,45.548,0.472,0.536,0.475,70,48,0,3.585
5,random,tac_600_1500,llama,shuffled,3,118,0.621,41.361,0.425,0.503,0.381,75,43,0,3.826
6,random,tac_600_1500,llama,shuffled,4,118,0.635,43.027,0.442,0.514,0.415,69,49,0,3.777
7,random,tac_600_1500,llama+minilm,shuffled,0,118,0.966,70.465,0.764,0.808,0.941,75,41,2,0.003
8,random,tac_600_1500,llama+minilm,shuffled,1,118,0.973,68.177,0.751,0.806,0.949,76,41,1,0.002
9,random,tac_600_1500,llama+minilm,shuffled,2,118,0.977,69.072,0.756,0.809,0.958,70,48,0,0.002
